In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from l3mezo import MeZOTrainer, MeZOTrainingArguments
from utils import ca_load_dataset
import torch
from datetime import datetime

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

model_base = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
model_train = AutoModelForCausalLM.from_pretrained('gpt2') # shouldn't to device? 
model_base.generation_config.pad_token_id = tokenizer.pad_token_id
model_train.generation_config.pad_token_id = tokenizer.pad_token_id

In [11]:
# NEED TO UPDATE THIS



training_args = MeZOTrainingArguments(
    output_dir=f'data/runs/L3_TRAIN/test-{str(datetime.now())}',
    evaluation_strategy='epoch',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f'logs/L3_TRAIN/test-{str(datetime.now())}',
    label_names=['corr_logits'], 
    lambda_train=0.6,
    original_model=model_base,  
    lr_scheduler_type='constant',
    learning_rate=1e-3,
    generation_max_length = 10, # set to a constant, matching the dataset processing   
)

# LR needs 2 be fixed

# Need to check if all data is correctly transfered to device (tokens, logits, etc)

# Need to check that inputs are correctly formatted and tokenized

c:\Users\nonam\miniconda3\envs\MAAA_CD\Lib\site-packages\transformers\training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [12]:
training_dataset = ca_load_dataset('greater_than_data.csv', model_base, tokenizer, device, max_new_tokens=10, batch_size=20)

In [15]:
trainer = MeZOTrainer(
    model=model_train, 
    args=training_args, 
    train_dataset=training_dataset['train'],
    tokenizer = tokenizer,
)

OSError: [WinError 123] The filename, directory name, or volume label syntax is incorrect: 'data/runs/L3_TRAIN/test-2025-08-07 15:25:09.721291'

In [ ]:
trainer.train()